# Tunable MZI in PhotonForge: the Precision with Light version
**Precision with Light, PhotonForge Series, Episode 3 (companion notebook)**

Based on the PhotonForge documentation example *Tunable MZI* (Flexcompute), using the open-source LNOI400 PDK from Luxtelligence. This version adds three things the original does not: **design numbers before simulation** (Vπ, fringe voltages), a **phase and transmission check against the analytic MZI**, and a **bandwidth check of the null** (what limits extinction across the C-band).

**What differs from the documentation example.** The original loads a phase shifter from `phase_shifter.gds`, produced in another example (Quantum Chip). So that this notebook runs on its own in Colab, we build a 100 µm heated waveguide with the PDK component `heated_straight`. Length and port specification match; the heater geometry does not, and since the heater law here is a placeholder anyway (see section 3), nothing downstream depends on it.

**Requirements.** A Flexcompute/Tidy3D account and API key. The mode solve and the 2x2 MMI run on the Tidy3D cloud and consume a small number of FlexCredits. Everything else is local.

*Status of this notebook:* the structure, the PDK calls, the netlist assembly and every NumPy cell were checked offline against the installed packages. The cloud-dependent cells follow the documentation example's calls; run it once top to bottom and compare your plots with the text of the article before publishing results.

## 0. Install dependencies (two steps, with a restart in between)

The Luxtelligence package itself does not pin NumPy; it only requires `photonforge>=2.0`. The NumPy/SciPy constraints come from PhotonForge and Tidy3D, and conflicts in Colab usually arise because an older NumPy is already loaded in the running session when pip upgrades it. So: install, **restart the session** (Runtime > Restart session), then continue from the version-check cell. Do not run the install cell again after the restart.

In [ ]:
# Step 1. Let pip's resolver choose a consistent set. Do NOT pin or reinstall numpy by hand.
!pip install -q "photonforge>=2.0" luxtelligence-lnoi400-forge ipywidgets
!pip check

**Now restart the session (Runtime > Restart session) and run from the next cell.**

In [ ]:
# Step 2. After the restart: confirm that a consistent stack is loaded.
import numpy, scipy, tidy3d, photonforge, luxtelligence_lnoi400_forge
from importlib.metadata import version
for p in ("numpy", "scipy", "tidy3d", "photonforge", "luxtelligence-lnoi400-forge"):
    print(f"{p:30s} {version(p)}")
# Tested offline with numpy 2.4.6, scipy (resolver choice), tidy3d 2.12.1,
# photonforge 2.0.0, luxtelligence-lnoi400-forge 2.1.1 on Python 3.12.

In [ ]:
# Tidy3D API key (needed for the mode solver and the MMI simulation).
# Get one at https://tidy3d.simulation.cloud/account?tab=apikey
import getpass, subprocess
key = getpass.getpass("Tidy3D API key: ")
subprocess.run(["tidy3d", "configure", "--apikey", key], check=True)
del key

## 1. Imports and defaults

In [ ]:
import luxtelligence_lnoi400_forge as lxt
import matplotlib.pyplot as plt
import numpy as np
import photonforge as pf
import tidy3d as td

td.config.logging.level = "ERROR"

# LiveViewer serves a local web page; it works on a desktop Jupyter, not in Colab.
try:
    import photonforge.live_viewer as live_viewer
    viewer = live_viewer.LiveViewer()
except Exception as exc:
    viewer = lambda c: print(f"(layout viewer unavailable here: {exc})")

In [ ]:
tech = lxt.lnoi400()
pf.config.default_technology = tech

# We only care about the fundamental TE mode
port_spec = tech.ports["RWG1000"]
port_spec.num_modes = 1

pf.config.default_kwargs = {
    "port_spec": port_spec,
    "radius": 75,
    "euler_fraction": 0.5,
}

wavelengths = np.linspace(1.5, 1.6, 21)   # um
lam0 = 1.55                                # um, centre wavelength
k0 = len(wavelengths) // 2                 # index of the centre wavelength

## 2. The phase shifter (stand-in for the GDSII in the original)

In [ ]:
L_HEATER = 100.0  # um
phase_shifter = lxt.component.heated_straight(length=L_HEATER, name="phase_shifter")
print("Ports:", list(phase_shifter.ports))
viewer(phase_shifter)

## 3. The heater law, and design numbers *before* any simulation

The example uses a deliberately simple law, `Δn = coefficient · V² / R`, with `R = 3 Ω` and `coefficient = 1e-3`. Because `V²/R` is the electrical power, the coefficient is an index change **per watt**. These are placeholders for measured or simulated data, not properties of a real LNOI400 heater. Keep that in mind when reading any absolute voltage below.

Inverse step first: for a heater of length L at wavelength λ, a phase change of π needs

**Δn_π = λ / (2 L)**, so **P_π = Δn_π / coefficient** and **V_π = √(P_π · R)**.

In [ ]:
R_HEATER = 3.0   # ohm   (placeholder from the documentation example)
K_TO     = 1e-3  # delta-n per watt (placeholder from the documentation example)

def index_change(voltage, resistance=R_HEATER, coefficient=K_TO):
    return coefficient * voltage**2 / resistance

dn_pi = lam0 / (2 * L_HEATER)
P_pi = dn_pi / K_TO
V_pi = np.sqrt(P_pi * R_HEATER)
print(f"delta-n for pi : {dn_pi:.5f}")
print(f"P_pi           : {P_pi:.2f} W   (placeholder law; a real heater needs mW-scale)")
print(f"V_pi           : {V_pi:.2f} V")
print("Voltages of the first four phase steps of pi (V_m = V_pi * sqrt(m)):",
      np.round(V_pi * np.sqrt(np.arange(1, 5)), 2))
print(f"Phase at 10 V  : {2*np.pi*index_change(10)*L_HEATER/lam0/np.pi:.2f} pi")

Read the last two lines as a prediction. Because the phase goes as V², the fringes get closer together as the voltage rises: expect the first transition near V_π and the following ones at V_π√2, V_π√3 and so on. If your plot in section 8 does not show that pattern, something in the model chain is wrong.

## 4. Effective index of the heated waveguide (cloud mode solve)

In [ ]:
mode_solver = pf.port_modes(port_spec, pf.C_0 / wavelengths, group_index=True)
_ = mode_solver.data.n_eff.plot()

## 5. The parametric heater model

`n_complex` is the baseline complex index versus wavelength; the voltage-dependent part is added on top. Note the modelling assumption inherited from the original: **Δn is the same at every wavelength**. Real thermo-optic coefficients vary a little over 100 nm.

In [ ]:
n_complex = mode_solver.data.n_complex.values.T

def index(voltage, resistance=R_HEATER, coefficient=K_TO):
    return n_complex + index_change(voltage, resistance, coefficient)

thermal_model = pf.WaveguideModel(n_complex=index(0))
phase_shifter.add_model(thermal_model, "Waveguide")

### Check 1: does the simulated phase match the analytic one?

The analytic phase of the heater section is Δφ = 2π Δn L / λ. The S-matrix phase sign depends on the time convention, so we compare magnitudes.

In [ ]:
voltages = np.linspace(0, 10, 51)
phases = []
for v in voltages:
    thermal_model.update(n_complex=index(v))
    s = phase_shifter.s_matrix(pf.C_0 / wavelengths, show_progress=False)
    phases.append(np.angle(s["P0@0", "P1@0"][k0]))
thermal_model.update(n_complex=index(0))   # reset

phi_sim = np.abs(np.unwrap(phases) - np.unwrap(phases)[0])
phi_ana = 2 * np.pi * index_change(voltages) * L_HEATER / lam0

plt.plot(voltages, phi_sim, label="PhotonForge (centre wavelength)")
plt.plot(voltages, phi_ana, "--", label="2π Δn L / λ")
plt.xlabel("Voltage (V)"); plt.ylabel("Phase change (rad)"); plt.legend()
plt.show()
print(f"max |difference| = {np.max(np.abs(phi_sim - phi_ana)):.3f} rad")

## 6. The 2x2 MMI

Taken from the Luxtelligence PDK, with port symmetries declared so that one FDTD run per input is not needed.

In [ ]:
mmi = lxt.component.mmi2x2_optimized1550(
    model=pf.Tidy3DModel(
        port_symmetries=[
            ("P1", "P0", "P3", "P2"),
            ("P2", "P3", "P0", "P1"),
            ("P3", "P2", "P1", "P0"),
        ]
    )
)
viewer(mmi)

In [ ]:
# Optional sanity check of the symmetries (coarser mesh, cloud runs)
assert mmi.active_model.test_port_symmetries(
    mmi, frequencies=pf.C_0 / wavelengths, grid_spec=8
)
s_mmi = mmi.s_matrix(pf.C_0 / wavelengths)
_ = pf.plot_s_matrix(s_mmi, input_ports=["P0"], output_ports=["P2", "P3"], y="dB")

## 7. Assembling the tunable MZI

Two MMIs and two heated arms. Both arms use the **same** `phase_shifter` component, which is why we cannot just change `thermal_model` for one arm: it would change both. The circuit model's `updates` argument solves this by applying a model update to one *reference* only, for the duration of that computation.

In [ ]:
netlist = {
    "name": "Tunable_MZI",
    "instances": {
        "mmi1": mmi,
        "ps0": {"component": phase_shifter, "origin": (280, -75)},
        "ps1": {"component": phase_shifter, "origin": (280, 75)},
        "mmi2": {"component": mmi, "origin": (580, 0)},
    },
    "routes": [
        (("mmi1", "P2"), ("ps0", "P0"), pf.parametric.route_s_bend),
        (("mmi1", "P3"), ("ps1", "P0"), pf.parametric.route_s_bend),
        (("mmi2", "P1"), ("ps1", "P1"), pf.parametric.route_s_bend),
        (("mmi2", "P0"), ("ps0", "P1"), pf.parametric.route_s_bend),
    ],
    "ports": [("mmi1", "P0"), ("mmi1", "P1"), ("mmi2", "P2"), ("mmi2", "P3")],
    "models": [(pf.CircuitModel(), "Circuit")],
}
mzi = pf.component_from_netlist(netlist)
viewer(mzi)

def drive(v0, v1):
    """Per-arm voltage -> circuit-model updates (layout and shared model untouched)."""
    return {
        ("phase_shifter", 0): {"model_updates": {"n_complex": index(v0)}},
        ("phase_shifter", 1): {"model_updates": {"n_complex": index(v1)}},
    }

def smat(v0, v1, freqs=None):
    freqs = pf.C_0 / wavelengths if freqs is None else freqs
    return mzi.s_matrix(freqs, model_kwargs={"updates": drive(v0, v1)}, show_progress=False)

s = smat(0, 0)
_ = pf.plot_s_matrix(s, input_ports=["P0"], output_ports=["P2", "P3"])

## 8. Power versus heater voltage (and the analytic overlay)

Same sweep as the documentation, at the centre wavelength, now with an **analytic overlay**. We normalise the two outputs by their sum to remove excess loss, read the zero-volt split to get the baseline phase φ0, and then draw `cos²((φ0 ± Δφ(V))/2)` with Δφ from section 3. Nothing is fitted except the baseline and the sign convention.

In [ ]:
v_sweep = np.linspace(0, 10, 101)
T = np.zeros((v_sweep.size, 2))
for i, v in enumerate(v_sweep):
    sv = smat(v, 0)
    T[i, 0] = np.abs(sv["P0@0", "P2@0"][k0]) ** 2
    T[i, 1] = np.abs(sv["P0@0", "P3@0"][k0]) ** 2

plt.plot(v_sweep, T, label=["|S₂₀|²", "|S₃₀|²"])
plt.xlabel("Voltage on phase shifter 0 (V)"); plt.ylabel("Transmission")
plt.legend(); plt.show()
print(f"Insertion loss at 0 V (centre wavelength): {-10*np.log10(T[0].sum()):.2f} dB")

In [ ]:
Tn = T / T.sum(axis=1, keepdims=True)            # loss-normalised
dphi = 2 * np.pi * index_change(v_sweep) * L_HEATER / lam0

# Output 'P3' behaves as cos^2(phase/2); try both sign conventions and both
# baseline signs, keep the best.
phi0_mag = 2 * np.arccos(np.sqrt(np.clip(Tn[0, 1], 0, 1)))
best = None
for sgn in (+1, -1):
    for phi0 in (+phi0_mag, -phi0_mag):
        model = np.cos((phi0 + sgn * dphi) / 2) ** 2
        err = np.sqrt(np.mean((model - Tn[:, 1]) ** 2))
        if best is None or err < best[0]:
            best = (err, sgn, phi0, model)
err, sgn, phi0, model = best

plt.plot(v_sweep, Tn[:, 1], label="PhotonForge (normalised |S₃₀|²)")
plt.plot(v_sweep, model, "--", label="analytic MZI")
for m in range(1, 5):
    plt.axvline(V_pi * np.sqrt(m), color="gray", lw=0.6, ls=":")
plt.xlabel("Voltage on phase shifter 0 (V)"); plt.ylabel("Normalised transmission")
plt.legend(); plt.show()
print(f"baseline phase φ0 = {phi0:.3f} rad, RMS difference = {err:.3f}")

## 9. Two heaters: one map instead of two sliders

The documentation adds interactive sliders for the two arms. Because the thermal phase is the difference of two quadratic terms, the whole behaviour is one picture: the output depends on **V0² − V1²** only (in this model, with no thermal crosstalk). The map below uses the analytic form validated in section 8, so it costs no simulations. Spot-check a few points against PhotonForge with `smat(v0, v1)`.

In [ ]:
va = np.linspace(0, 10, 201)
V0, V1 = np.meshgrid(va, va)
dphi2 = 2 * np.pi * (index_change(V0) - index_change(V1)) * L_HEATER / lam0
T3 = np.cos((phi0 + sgn * dphi2) / 2) ** 2

plt.figure(figsize=(5.5, 4.6))
plt.pcolormesh(V0, V1, T3, shading="auto", cmap="viridis")
plt.colorbar(label="Normalised |S₃₀|² (analytic)")
plt.xlabel("V0 (V), arm 0"); plt.ylabel("V1 (V), arm 1")
plt.plot([0, 10], [0, 10], "w--", lw=0.8)  # V0 = V1: no net phase
plt.title("Same colour along V0² − V1² = const")
plt.show()

# spot check against the simulator
for (a, b) in [(6, 1), (4, 4), (7, 3)]:
    sv = smat(a, b)
    t2, t3 = (np.abs(sv["P0@0", p][k0]) ** 2 for p in ("P2@0", "P3@0"))
    ana = np.cos((phi0 + sgn * 2*np.pi*(index_change(a)-index_change(b))*L_HEATER/lam0) / 2) ** 2
    print(f"V0={a}, V1={b}: PhotonForge {t3/(t2+t3):.3f}  analytic {ana:.3f}")

## 10. How dark can the null be across the band?

The heater sets the phase at one wavelength; the phase it produces scales as 1/λ. If you set Δφ = π at 1.55 µm, then at wavelength λ the phase is π·(1.55/λ), and the leakage into the nominally dark port is sin²(δ/2) with δ = π·(1.55/λ − 1). That is a floor on the extinction ratio that exists even with perfect 50:50 MMIs.

In [ ]:
lam_band = np.linspace(1.5, 1.6, 201)
delta = np.pi * (lam0 / lam_band - 1)
leak = np.sin(delta / 2) ** 2
er_floor_db = -10 * np.log10(np.maximum(leak, 1e-12))
print(f"Extinction-ratio floor from the 1/λ phase alone: "
      f"{er_floor_db[0]:.1f} dB at 1.50 µm, {er_floor_db[-1]:.1f} dB at 1.60 µm")

plt.plot(lam_band * 1e3, er_floor_db)
plt.ylim(10, 60); plt.xlabel("Wavelength (nm)")
plt.ylabel("ER floor (dB)"); plt.title("Null set at 1550 nm, perfect MMIs")
plt.show()

In [ ]:
# Now the real device: find the first null at the centre wavelength, then look at the whole band.
i_null = np.argmin(Tn[: np.searchsorted(v_sweep, 1.3 * V_pi), 1] if Tn[0, 1] > 0.5
                   else -Tn[: np.searchsorted(v_sweep, 1.3 * V_pi), 1])
v_null = v_sweep[i_null]
print(f"First null at {v_null:.2f} V (design number V_π = {V_pi:.2f} V)")

sv = smat(v_null, 0)
t2 = np.abs(sv["P0@0", "P2@0"]) ** 2
t3 = np.abs(sv["P0@0", "P3@0"]) ** 2
dark, bright = (t3, t2) if Tn[0, 1] > 0.5 else (t2, t3)
er_sim = 10 * np.log10(bright / dark)

plt.plot(wavelengths * 1e3, er_sim, "o-", label="PhotonForge, V at 1550 nm null")
plt.plot(lam_band * 1e3, er_floor_db, "--", label="1/λ floor (ideal MMIs)")
plt.xlabel("Wavelength (nm)"); plt.ylabel("Extinction ratio (dB)")
plt.legend(); plt.show()

## 11. Export (from a clean component)

Here, unlike the passive MZI episode, the layout was never edited in place: all voltage changes went through `model_kwargs`. The component you export is the one you drew.

In [ ]:
pf.write_layout("tunable_mzi.gds", mzi)
pf.write_phf("tunable_mzi.phf", mzi)
print("Wrote tunable_mzi.gds and tunable_mzi.phf")

## Where to go next
- Replace the placeholder heater law with measured or simulated data (the documentation points to a Tidy3D metal-heater phase-shifter notebook) and re-check Section 3's design numbers.
- Add thermal crosstalk between the two arms: the V0² − V1² result of Section 9 stops being exact.
- Add a transient: heater response is a thermal time constant, not an instantaneous index change.
- Vary the process: corner analysis for this device is the subject of the next notebook.

*Source: PhotonForge documentation, "Tunable MZI" (Flexcompute). Flexcompute, Tidy3D and PhotonForge are products of Flexcompute Inc. Luxtelligence LNOI400 PDK: luxtelligence-lnoi400-forge on PyPI.*